# NILM – Semaines 3-4 : réseau de neurones, matrices de confusion et flexibilité

**Objectif.** Ce notebook reprend le réseau de neurones de la semaine 3 dans sa **version de base** (32 canaux, 22 120 paramètres) et le compare au **CO** et au **naïf** sur **exactement le même test**. La semaine 4 y ajoute :
- l'analyse détaillée des erreurs avec des **matrices de confusion** et l'effet du seuil de décision (section 10) ;
- une **simulation de flexibilité** : pilotage d'un chauffe-eau et d'une batterie selon le prix de l'électricité, en 4 scénarios et avec deux tarifs (section 11) ;
- l'**apport de la désagrégation** pour les appareils décalables, lave-linge et lave-vaisselle (section 12).

**Données.** UK-DALE maison 1 sur un an (dossier `an1`, moyenné à la minute), exécuté sous Colab avec les fichiers dans `/content` (GPU T4 conseillé, le CPU fonctionne aussi).

**Ce que fait le réseau.** Il reçoit la courbe totale de la maison sur une longue fenêtre (environ 4 heures de contexte autour de chaque minute) et en tire, pour chaque appareil : (1) une puissance, (2) une probabilité d'être allumé. Là où le CO ne voit que la valeur de la minute, le réseau voit la *forme* de la courbe avant et après.

**Règle de décision du projet :** je ne garde le réseau que s'il bat le CO sur ce test.

## 1. Réglages
- `MAINS`, `APPLIANCES` : le canal de la maison entière et les 4 appareils (frigo, bouilloire, lave-linge, lave-vaisselle) ;
- `LOAD` : un an de données, du 1er mars 2013 au 28 février 2014, au pas d'une minute (`RESAMPLE`) ;
- `ON_W` : seuils « allumé » (30 W, 500 W, 20 W, 20 W) et `N_ON_STATES` : niveaux de puissance du CO ;
- `TEST_FRAC`, `VAL_FRAC` : 30 % de la fin pour le test, les 7 % juste avant pour la validation ;
- `USE_TIME` : ajoute l'heure de la journée aux entrées ;
- `CROP`, `BATCH`, `STEPS`, `EPOCHS`, `PATIENCE` : morceaux de 12 h, lots de 32, 100 pas par époque, 40 époques au plus, arrêt après 8 époques sans progrès ;
- `CH`, `LR` : 32 canaux internes, pas d'apprentissage de 10⁻³.

In [ ]:
from pathlib import Path
from itertools import product
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# ================== RÉGLAGES (à adapter) ==================
DATA_DIR = Path("/content")
MAINS = "aggregate"
APPLIANCES = ["fridge", "kettle", "washing_machine", "dishwasher"]
LOAD = ("2013-03-01", "2014-02-28")          # fenêtre de dates lue dans les fichiers
RESAMPLE = "1min"
ON_W = {"fridge": 30, "kettle": 500, "washing_machine": 20, "dishwasher": 20}    # seuil « allumé » (W)
N_ON_STATES = {"fridge": 1, "kettle": 1, "washing_machine": 3, "dishwasher": 3}  # niveaux du CO
TEST_FRAC, VAL_FRAC = 0.30, 0.07             # fin des données = test ; juste avant = validation
USE_TIME = True                              # ajouter l'heure de la journée comme entrée du réseau
CROP, BATCH, STEPS, EPOCHS, PATIENCE = 720, 32, 100, 40, 8   # morceaux de 12 h ; 100 pas par époque
CH, LR = 32, 1e-3                            # largeur du réseau, pas d'apprentissage
# ===========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("calcul sur :", device, "(GPU conseillé)" if device == "cpu" else "")
torch.manual_seed(0); np.random.seed(0)
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True

## 2. Chargement, grille de temps complète et découpe
Je lis les fichiers UK-DALE de la maison 1 : `labels.dat` me donne le numéro de canal de chaque appareil, puis je lis un `channel_N.dat` par canal (horodatage Unix + puissance) sur un an, du 1er mars 2013 au 28 février 2014. Chaque canal est ramené à **un point par minute** (moyenne des mesures à 6 s).

Je recale ensuite tous les canaux sur **une grille de temps complète** (une ligne par minute, même quand une mesure manque). C'est indispensable pour le réseau, qui lit des fenêtres continues de plusieurs heures. Une minute n'est déclarée **valide** que si tous les canaux ont une mesure ; dans les trous, les entrées sont mises à zéro et ces minutes ne comptent ni dans la perte ni dans les scores (masque `valid`).

La découpe est **chronologique**, sans mélange aléatoire (sinon des minutes voisines se retrouveraient des deux côtés et le test serait trop facile) :
- **entraînement** : le début de l'année, du 1er mars au 16 octobre 2013 (≈ 312 000 minutes valides) ;
- **validation** : les 7 % suivants, jusqu'au 11 novembre 2013 (≈ 36 000 minutes), qui servent uniquement à décider quand arrêter l'entraînement ;
- **test** : les 30 % de la fin, du 11 novembre 2013 au 28 février 2014 (≈ 152 700 minutes valides), jamais vus pendant l'entraînement. Tous les scores du notebook sont calculés là.

Sur l'année, 95,2 % des minutes sont valides.

In [ ]:
def read_labels(folder):
    labels = {}
    for line in (folder / "labels.dat").read_text().splitlines():
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            labels[parts[1].strip()] = int(parts[0])
    return labels

def read_channel(folder, ch):
    df = pd.read_csv(folder / f"channel_{ch}.dat", sep=r"\s+", header=None, usecols=[0, 1], names=["t", "w"])
    df.index = pd.to_datetime(df["t"], unit="s")
    s = df["w"].astype(float)
    s = s[~s.index.duplicated()].sort_index()
    return s.loc[LOAD[0]:LOAD[1]]

labels = read_labels(DATA_DIR)
series = {}
for name in [MAINS] + APPLIANCES:
    if name not in labels:
        print("absent de labels.dat :", name); continue
    try:
        s = read_channel(DATA_DIR, labels[name])
    except FileNotFoundError:
        print(f"fichier manquant : channel_{labels[name]}.dat ({name})"); continue
    if s.empty:
        print("aucune donnée sur la période pour", name); continue
    series[name] = s.resample(RESAMPLE).mean()
    print(f"chargé : {name:16s} {len(s):>8} points  {s.index.min()} -> {s.index.max()}")
if MAINS not in series:
    raise SystemExit("Canal maison entière introuvable : vérifie les fichiers envoyés dans /content.")
APPLIANCES = [a for a in APPLIANCES if a in series]
n_app = len(APPLIANCES)

data = pd.DataFrame(series)
ends = {n: sr.index.max() for n, sr in series.items()}
end_common = min(ends.values())
if max(ends.values()) - end_common > pd.Timedelta(days=7):
    print("\nATTENTION : les canaux ne finissent pas à la même date (fichiers peut-être incomplets, envoi interrompu ?) :")
    for n, e in ends.items():
        print(f"   {n:16s} -> {e}")
    print("On garde seulement la période commune, jusqu'au", end_common)
data = data.loc[:end_common]
if end_common - data.index.min() < pd.Timedelta(days=60):
    print("ATTENTION : moins de 2 mois de données communes, le réseau sera peu fiable.")
grid = pd.date_range(data.index.min(), data.index.max(), freq=RESAMPLE)
data = data.reindex(grid)                                  # grille régulière : les trous deviennent NaN
valid = data.notna().all(axis=1).values                    # minutes où tous les canaux ont une mesure
T = len(data)
print(f"\nminutes valides : {valid.sum()} / {T} ({100 * valid.mean():.1f} %)")

agg = data[MAINS].fillna(0).values.astype(np.float32)
Yw = data[APPLIANCES].fillna(0).values.astype(np.float32).T     # (n_app, T) en W
thr = np.array([ON_W[a] for a in APPLIANCES], dtype=np.float32)

i_test = int(T * (1 - TEST_FRAC)); i_val = int(T * (1 - TEST_FRAC - VAL_FRAC))
for name, a, b in [("entraînement", 0, i_val), ("validation", i_val, i_test), ("test", i_test, T)]:
    print(f"{name:13s}: {data.index[a]} -> {data.index[b - 1]}  ({valid[a:b].sum()} minutes valides)")
tr = valid.copy(); tr[i_val:] = False                       # minutes valides d'entraînement
te = valid.copy(); te[:i_test] = False                      # minutes valides de test
if tr.sum() < 1000 or te.sum() < 1000:
    raise SystemExit("Pas assez de données valides : vérifie LOAD et les fichiers envoyés.")

## 3. Préparation : consommation de fond, échelles, entrées du réseau
- **Consommation de fond** : médiane de (total − somme des 4 appareils) sur l'entraînement. Elle représente tout ce que je ne suis pas (éclairage, TV, box, veilles, four…). Je l'estime à environ **178 W** ; elle sert au CO, qui la retire du total avant de chercher les appareils. Les 4 appareils suivis ne représentent qu'environ **27 %** de l'énergie de la maison : le modèle doit donc apprendre à ignorer les 73 % restants.
- **Échelle de chaque appareil** : sa puissance « haute » (99e centile des minutes où il est allumé) : environ 250 W pour le frigo, 2 405 W pour la bouilloire, 1 975 W pour le lave-linge et 2 392 W pour le lave-vaisselle. Je divise les puissances cibles par ces échelles pour que le réseau travaille avec des valeurs de l'ordre de 1 ; à la sortie, je remultiplie.
- **États allumé / éteint** : un appareil est considéré allumé au-dessus d'un seuil fixe (`ON_W` : 30 W pour le frigo, 500 W pour la bouilloire, 20 W pour les lave-linge et lave-vaisselle). Sur l'entraînement, le frigo est allumé 43 % du temps, mais la bouilloire moins de 1 %, le lave-linge 5 % et le lave-vaisselle 2 % : ce déséquilibre est la principale difficulté.
- **Entrées du réseau** (3 canaux par minute) : la puissance totale de la maison en **kW**, puis l'heure de la journée codée en **sinus et cosinus** (`USE_TIME`). Ce codage circulaire évite la coupure artificielle entre 23 h 59 et 0 h 00.

In [ ]:
offset = float(np.median(np.clip(agg[tr] - Yw[:, tr].sum(0), 0, None)))
print(f"Consommation de fond estimée : {offset:.0f} W")
print(f"Part de l'énergie venant des {n_app} appareils suivis : {Yw[:, tr].sum() / agg[tr].sum() * 100:.0f} %")

scale_y = np.array([np.percentile(Yw[i, tr & (Yw[i] > thr[i])], 99) if (tr & (Yw[i] > thr[i])).any() else 1.0
                    for i in range(n_app)], dtype=np.float32)
print("échelles (W) :", dict(zip(APPLIANCES, scale_y.round(0).tolist())))

rows = [agg / 1000.0]
if USE_TIME:
    hour = (data.index.hour + data.index.minute / 60).values
    rows += [np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24)]
X = np.vstack(rows).astype(np.float32)
X[:, ~valid] = 0.0                                          # entrées neutres dans les trous
Y = (Yw / scale_y[:, None]).astype(np.float32)
ON = (Yw > thr[:, None]).astype(np.float32)
M = valid.astype(np.float32)
prev = ON[:, tr].mean(1)
print("part du temps allumé (entraînement) :", dict(zip(APPLIANCES, prev.round(3).tolist())))

## 4. Les deux références : naïf et CO (comme en semaine 2)
Pour juger le réseau, je garde les deux références de la semaine 2, recalculées sur **exactement la même découpe** :
- **Naïf** (« toujours allumé ») : le plancher. Un modèle qui ne fait pas mieux n'apporte rien.
- **Optimisation combinatoire (CO)** : les niveaux de puissance de chaque appareil sont appris (k-means à une dimension) **sur la période d'entraînement seulement** (1 niveau pour le frigo et la bouilloire, 3 pour le lave-linge et le lave-vaisselle), ce qui donne 64 combinaisons d'états. À chaque minute du test, je garde la combinaison dont la somme est la plus proche du total moins la consommation de fond.

In [ ]:
def kmeans1d(x, k, iters=100):
    if len(x) > 50000:
        x = np.random.default_rng(0).choice(x, 50000, replace=False)
    c = np.quantile(x, np.linspace(0.1, 0.9, k)) if k > 1 else np.array([x.mean()])
    for _ in range(iters):
        lab = np.abs(x[:, None] - c[None, :]).argmin(1)
        new = np.array([x[lab == j].mean() if (lab == j).any() else c[j] for j in range(k)])
        if np.allclose(new, c):
            break
        c = new
    return np.sort(c)

states = {}
for i, a in enumerate(APPLIANCES):
    on = Yw[i][tr & (Yw[i] > thr[i])]
    if len(on) < 10:
        states[a] = np.array([0.0]); continue
    k = min(N_ON_STATES[a], max(1, len(np.unique(on.round()))))
    states[a] = np.concatenate([[0.0], kmeans1d(on, k)])

combo_states = np.array(list(product(*[states[a] for a in APPLIANCES])))
combo_sums = combo_states.sum(axis=1)
idx_te = np.where(te)[0]
y_in = np.clip(agg[idx_te] - offset, 0, None)
best = np.empty(len(idx_te), dtype=int)
for i in range(0, len(idx_te), 20000):
    best[i:i + 20000] = np.abs(y_in[i:i + 20000, None] - combo_sums[None, :]).argmin(1)
pred_co = np.zeros((n_app, T), dtype=np.float32)
pred_co[:, idx_te] = combo_states[best].T
on_co = pred_co > thr[:, None]

mean_on = np.array([Yw[i, tr & (Yw[i] > thr[i])].mean() if (tr & (Yw[i] > thr[i])).any() else 0.0 for i in range(n_app)], dtype=np.float32)
pred_naive = np.repeat(mean_on[:, None], T, axis=1)
on_naive = np.ones((n_app, T), dtype=bool)
print("références prêtes :", len(combo_sums), "combinaisons pour le CO")

## 5. Le réseau
Mon réseau est un **réseau convolutif 1D dilaté à connexions résiduelles** :
- une couche d'entrée (convolution 1×1) qui passe des 3 canaux d'entrée à 32 canaux internes ;
- **7 blocs** de convolution de noyau 3 avec des dilatations **1, 2, 4, 8, 16, 32 et 64 minutes**. Chaque bloc ajoute son résultat à son entrée (connexion résiduelle), ce qui facilite l'apprentissage ;
- une couche de sortie (convolution 1×1) qui donne **8 valeurs par minute** : 4 puissances normalisées (une par appareil) et 4 scores « allumé » (logits).

Grâce aux dilatations, chaque minute de sortie dépend de **127 minutes avant et 127 minutes après** (champ réceptif d'environ 4 h 15). C'est ce qui lui permet de reconnaître la *forme* d'un cycle (chauffe d'un lave-linge, impulsion de la bouilloire) là où le CO ne voit qu'une valeur instantanée. Comme il est entièrement convolutif, il accepte une fenêtre de n'importe quelle longueur.

Il ne compte que **22 120 paramètres** (≈ 86 Ko en float32, ≈ 22 Ko en int8) : c'est volontaire, il doit tenir et tourner sur le microcontrôleur STM32 côté capteur.

**Fonction de perte** (la somme de deux termes) :
- **puissance** : erreur *smooth L1* entre puissance prédite et réelle (normalisées), calculée **seulement quand l'appareil est allumé** et moyennée appareil par appareil, pour que le frigo, toujours présent, n'écrase pas les autres ;
- **allumé / éteint** : entropie croisée binaire, avec un poids plus fort (jusqu'à ×10) sur les minutes « allumé » des appareils rares, sinon le réseau aurait intérêt à toujours répondre « éteint ».

In [ ]:
class Net(nn.Module):
    def __init__(self, n_in, n_out, ch=32, dils=(1, 2, 4, 8, 16, 32, 64)):
        super().__init__()
        self.inp = nn.Conv1d(n_in, ch, 1)
        self.blocks = nn.ModuleList([nn.Conv1d(ch, ch, 3, padding=d, dilation=d) for d in dils])
        self.out = nn.Conv1d(ch, n_out, 1)
    def forward(self, x):                       # x : (lot, n_in, temps) -> (lot, n_out, temps)
        h = F.relu(self.inp(x))
        for b in self.blocks:
            h = h + F.relu(b(h))
        return self.out(h)

net = Net(X.shape[0], 2 * n_app, CH).to(device)
n_par = sum(p.numel() for p in net.parameters())
print(f"{n_par} paramètres  ≈ {n_par * 4 / 1024:.0f} Ko en float32, ≈ {n_par / 1024:.0f} Ko en int8 (cible STM32)")

Xt, Yt, ONt = (torch.tensor(a, device=device) for a in (X, Y, ON))
Mt = torch.tensor(M, device=device)
pos_w = torch.tensor(np.clip((1 - prev) / np.maximum(prev, 1e-4), 1, 10), dtype=torch.float32, device=device).view(-1, 1)

def loss_fn(out, y, on, m):
    # out : (lot, 2*n_app, temps) ; y, on : (lot, n_app, temps) ; m : (lot, 1, temps)
    reg, logit = out[:, :n_app], out[:, n_app:]
    # puissance : apprise seulement quand l'appareil est allumé, normalisée appareil par appareil
    num = (F.smooth_l1_loss(reg, y, reduction="none") * on * m).sum(dim=(0, 2))
    den = (on * m).sum(dim=(0, 2)) + 1.0
    l_reg = (num / den).mean()
    # allumé / éteint : classification avec poids pour les appareils rarement allumés
    l_cls = (F.binary_cross_entropy_with_logits(logit, on, pos_weight=pos_w, reduction="none") * m).sum() / (m.sum() * n_app + 1e-6)
    return l_reg + l_cls

def get_batch(lo, hi):
    starts = np.random.randint(lo, hi - CROP, BATCH)
    idx = torch.tensor(starts[:, None] + np.arange(CROP)[None, :], device=device)    # (lot, temps)
    x = Xt[:, idx].permute(1, 0, 2)
    y = Yt[:, idx].permute(1, 0, 2)
    on = ONt[:, idx].permute(1, 0, 2)
    m = Mt[idx].unsqueeze(1)
    return x, y, on, m

@torch.no_grad()
def run(a, b, ctx=512):
    """Sortie du réseau sur [a, b), avec ctx minutes de contexte avant a."""
    lo = max(0, a - ctx)
    out = net(Xt[:, lo:b].unsqueeze(0))[0]
    return out[:, a - lo:]

def val_loss():
    net.eval()
    out = run(i_val, i_test).unsqueeze(0)
    l = loss_fn(out, Yt[:, i_val:i_test].unsqueeze(0), ONt[:, i_val:i_test].unsqueeze(0), Mt[i_val:i_test].view(1, 1, -1)).item()
    net.train()
    return l

## 6. Entraînement
À chaque époque, je fais 100 pas d'optimisation (Adam, pas d'apprentissage 10⁻³). À chaque pas, je tire au hasard **32 morceaux de 12 h** (720 minutes) dans la période d'entraînement UK-DALE. Travailler sur des morceaux tirés au hasard plutôt que sur l'année entière donne beaucoup de variété au réseau.

À la fin de chaque époque, je calcule la perte sur la **validation** (la tranche UK-DALE réelle juste avant le test). Si elle ne baisse plus pendant 8 époques (`PATIENCE`), j'arrête : continuer reviendrait à apprendre les mois d'entraînement par cœur (sur-apprentissage). Je recharge alors les poids de **la meilleure époque** et je les sauvegarde dans `nilm_cnn.pt`. Sur le GPU du Mac (`mps`), l'entraînement complet prend moins d'une minute.

La courbe d'apprentissage affichée ensuite montre les deux pertes : tant qu'elles baissent ensemble, le réseau apprend des choses qui se généralisent ; si la validation remontait pendant que l'entraînement continue de baisser, ce serait du sur-apprentissage.

In [ ]:
opt = torch.optim.Adam(net.parameters(), lr=LR)
best_val, best_state, bad, hist = 1e9, None, 0, []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    net.train(); tot = 0.0
    for _ in range(STEPS):
        x, y, on, m = get_batch(0, i_val)
        loss = loss_fn(net(x), y, on, m)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    vl = val_loss(); hist.append((tot / STEPS, vl))
    print(f"époque {ep:2d}  perte entraînement {tot / STEPS:.4f}  validation {vl:.4f}  ({time.time() - t0:.0f} s)")
    if vl < best_val - 1e-4:
        best_val, bad = vl, 0
        best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE:
            print("arrêt : plus de progrès sur la validation"); break
net.load_state_dict(best_state)
torch.save(best_state, "nilm_cnn.pt")
print("meilleure perte de validation :", round(best_val, 4), "| poids sauvegardés dans nilm_cnn.pt")

In [ ]:
h = np.array(hist)
fig, ax = plt.subplots()
ax.plot(h[:, 0], label="entraînement"); ax.plot(h[:, 1], label="validation")
ax.set_xlabel("époque"); ax.set_ylabel("perte"); ax.set_title("Courbe d'apprentissage (si la validation remonte : sur-apprentissage)")
ax.legend(); plt.show()

## 7. Prédiction sur le test
Je fais passer toute la période de test dans le réseau en une seule fois, avec 512 minutes de contexte avant le début du test. Pour chaque appareil et chaque minute :
- la **probabilité d'être allumé** est obtenue avec une sigmoïde sur le score ; au-dessus de 0,5, l'appareil est déclaré allumé ;
- la **puissance** est la sortie de régression (partie positive) remultipliée par l'échelle de l'appareil, et mise à 0 quand le réseau dit « éteint ».

In [ ]:
net.eval()
out = run(i_test, T).cpu().numpy()
reg = np.maximum(out[:n_app], 0) * scale_y[:, None]
p_on = 1 / (1 + np.exp(-out[n_app:]))
on_nn = np.zeros((n_app, T), dtype=bool); on_nn[:, i_test:] = p_on > 0.5
pred_nn = np.zeros((n_app, T), dtype=np.float32); pred_nn[:, i_test:] = reg * (p_on > 0.5)
print("prédiction terminée sur", te.sum(), "minutes de test valides")

## 8. Comparaison : le tableau à retenir
J'utilise les mêmes métriques qu'en semaine 2, calculées **uniquement sur les minutes valides de la période de test** :
- **F1** (0 à 1) : l'appareil est-il détecté allumé au bon moment ? Il combine la **précision** (quand le modèle dit « allumé », a-t-il raison ?) et le **rappel** (retrouve-t-il tous les moments où l'appareil tourne ?) ;
- **MAE (W)** : écart moyen, minute par minute, entre puissance prédite et réelle ;
- **erreur d'énergie (%)** : écart relatif sur les kWh attribués à l'appareil sur toute la période de test. C'est la métrique la plus utile pour la facture et pour la flexibilité.

Pour chaque appareil, la comparaison qui compte est **Réseau contre CO**. Le second tableau résume le gain de F1 et les erreurs d'énergie des deux méthodes.

In [ ]:
dt_h = pd.Timedelta(RESAMPLE).total_seconds() / 3600

def metrics(y_true, y_pred, on_pred, thr_a):
    t, p = y_true > thr_a, on_pred
    tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
    e_t, e_p = y_true.sum() * dt_h / 1000, y_pred.sum() * dt_h / 1000
    return {
        "F1": 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan,
        "précision": tp / (tp + fp) if (tp + fp) else np.nan,
        "rappel": tp / (tp + fn) if (tp + fn) else np.nan,
        "MAE (W)": float(np.abs(y_true - y_pred).mean()),
        "kWh réels": e_t, "kWh prédits": e_p,
        "erreur énergie (%)": 100 * abs(e_p - e_t) / e_t if e_t > 0 else np.nan,
    }

rows = []
for method, pred, on in [("Naïf", pred_naive, on_naive), ("CO", pred_co, on_co), ("Réseau", pred_nn, on_nn)]:
    for i, a in enumerate(APPLIANCES):
        rows.append({"appareil": a, "méthode": method, **metrics(Yw[i, te], pred[i, te], on[i, te], thr[i])})
results = pd.DataFrame(rows).set_index(["appareil", "méthode"]).sort_index()
results.round(2)

In [ ]:
f1 = results["F1"].unstack("méthode")[["Naïf", "CO", "Réseau"]]
ax = f1.plot.bar(rot=0, color=["#B0BEC5", "#D99200", "#0E7C86"], title="F1 par appareil (période de test)", ylim=(0, 1))
ax.set_ylabel("F1 (1 = parfait)"); plt.show()

cmp = pd.DataFrame({"F1 CO": f1["CO"], "F1 réseau": f1["Réseau"], "gain F1": f1["Réseau"] - f1["CO"],
                    "erreur énergie CO (%)": results.xs("CO", level="méthode")["erreur énergie (%)"],
                    "erreur énergie réseau (%)": results.xs("Réseau", level="méthode")["erreur énergie (%)"]})
cmp.round(2)

## 9. Une journée de test : réel, CO, réseau
Pour voir concrètement le comportement des modèles, j'affiche la journée de test où les 4 appareils consomment le plus d'énergie (le 31 décembre 2013 dans mes exécutions). Pour chaque appareil : la puissance réelle mesurée par sa prise, celle du CO et celle du réseau. On voit si les cycles sont trouvés au bon moment, s'ils ont la bonne durée et si des cycles sont inventés.

In [ ]:
idx_t = data.index[i_test:]
vt = valid[i_test:]
day_energy = pd.Series(Yw[:, i_test:].sum(0)[vt], index=idx_t[vt]).groupby(idx_t[vt].date).sum()
day = str(day_energy.idxmax())                         # jour de test avec le plus d'activité
print("jour affiché :", day)
sel = np.array([str(d.date()) == day for d in data.index])
t_ax = data.index[sel]
fig, axes = plt.subplots(n_app, 1, figsize=(11, 2.3 * n_app), sharex=True)
for ax, i, a in zip(np.atleast_1d(axes), range(n_app), APPLIANCES):
    ax.plot(t_ax, Yw[i, sel], color="#D99200", label="réel")
    ax.plot(t_ax, pred_co[i, sel], color="#8E9AAF", label="CO", alpha=0.8)
    ax.plot(t_ax, pred_nn[i, sel], color="#0E7C86", label="réseau", alpha=0.9)
    ax.set_title(a + " (W)", fontsize=10); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

## 10. Matrices de confusion : ce que le modèle réussit… et rate

**Comment je lis une matrice de confusion.** Pour **un** appareil, je prends chaque minute du test et je regarde deux choses : *était-il vraiment allumé ?* (la réalité, mesurée par la prise de l'appareil) et *le modèle a-t-il dit « allumé » ?* (sa prédiction). Cela donne 4 cas :

| | le modèle dit « éteint » | le modèle dit « allumé » |
|---|---|---|
| **réellement éteint** | ✅ **bien éteint** (vrai négatif) | ❌ **fausse alerte** (faux positif) |
| **réellement allumé** | ❌ **raté** (faux négatif) | ✅ **bien détecté** (vrai positif) |

Sur le graphique, **chaque ligne est ramenée à 100 %**, sinon on ne verrait rien : un lave-vaisselle est éteint plus de 95 % du temps. La case en bas à droite est le **rappel** (parmi les minutes où l'appareil tourne vraiment, quelle part le modèle retrouve) et la case en haut à droite le **taux de fausses alertes**. Une bonne matrice a une diagonale foncée (haut-gauche et bas-droite) et des cases claires ailleurs.

**Pourquoi 4 matrices 2×2 et pas une grande ?** Plusieurs appareils peuvent tourner en même temps : ce n'est pas un choix « un appareil parmi quatre », c'est une question oui/non posée séparément pour chaque appareil.

**Ce que fait le modèle, en clair.** Pour chaque minute, il regarde la courbe de la maison sur environ 4 heures autour d'elle et sort, pour chaque appareil, une **probabilité d'être allumé**. Au-dessus de **50 %**, il dit « allumé ». Le CO, lui, ne regarde que la valeur de la minute et choisit la combinaison d'appareils qui s'en rapproche le plus.

In [ ]:
def confusion(y_true, on_pred, thr_a):
    # lignes : réalité (éteint, allumé) ; colonnes : prédiction (éteint, allumé)
    t, p = y_true > thr_a, on_pred
    return np.array([[(~t & ~p).sum(), (~t & p).sum()],
                     [(t & ~p).sum(),  (t & p).sum()]])

fig, axes = plt.subplots(2, n_app, figsize=(3.3 * n_app, 6.4))
axes = np.array(axes).reshape(2, n_app)
cms = {}
for r, (method, on) in enumerate([("CO", on_co), ("Réseau", on_nn)]):
    for i, a in enumerate(APPLIANCES):
        cm = confusion(Yw[i, te], on[i, te], thr[i]); cms[(method, a)] = cm
        pct = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
        ax = axes[r, i]
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
        for u in range(2):
            for v in range(2):
                ax.text(v, u, f"{pct[u, v]:.0f} %\n({cm[u, v]:,})".replace(",", " "), ha="center", va="center",
                        fontsize=9, color="white" if pct[u, v] > 55 else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["prédit\néteint", "prédit\nallumé"], fontsize=8)
        ax.set_yticks([0, 1]); ax.set_yticklabels(["réel\néteint", "réel\nallumé"], fontsize=8)
        ax.set_title(f"{a}\n{method}", fontsize=10); ax.grid(False)
plt.tight_layout(); plt.show()

conf_rows = []
for (method, a), cm in cms.items():
    conf_rows.append({"appareil": a, "méthode": method, "bien éteint": cm[0, 0], "fausses alertes": cm[0, 1],
                      "ratés": cm[1, 0], "bien détecté": cm[1, 1]})
pd.DataFrame(conf_rows).set_index(["appareil", "méthode"]).sort_index()

### Ce que ça veut dire, appareil par appareil
La cellule suivante traduit automatiquement chaque matrice en phrases, à partir des chiffres :
- **Rappel** : sur 100 minutes où l'appareil tourne vraiment, combien le modèle en retrouve ;
- **Précision** : sur 100 minutes où le modèle annonce « allumé », combien sont vraies ;
- **Fausses alertes** : part des minutes éteintes où le modèle se trompe.

Ces deux grandeurs ne s'améliorent pas ensemble : un modèle qui dit « allumé » à tout va a un rappel parfait et une précision très faible (c'est exactement le modèle naïf).

In [ ]:
def fmt(x, pct=True):
    return "n/a" if not np.isfinite(x) else (f"{100 * x:.0f} %" if pct else f"{x:.1f}")

for a in APPLIANCES:
    print(f"=== {a} ===")
    for method in ["CO", "Réseau"]:
        tn, fp, fn, tp = cms[(method, a)].ravel()
        rappel = tp / (tp + fn) if tp + fn else np.nan
        prec = tp / (tp + fp) if tp + fp else np.nan
        fa = fp / (fp + tn) if fp + tn else np.nan
        k_ = lambda v: f"{v:,}".replace(",", " ")
        print(f"  {method:7s} retrouve {fmt(rappel)} des minutes où l'appareil tourne ; "
              f"quand il dit « allumé », il a raison {fmt(prec)} du temps ; "
              f"{k_(fp)} fausses alertes ({fmt(fa)} des minutes éteintes), {k_(fn)} minutes ratées.")

### Le seuil de décision (le « 50 % » du réseau)
Le réseau sort une probabilité ; c'est moi qui fixe à partir de quand je dis « allumé ». **Baisser le seuil** fait détecter plus de cycles (rappel ↑) mais avec plus d'erreurs (précision ↓) ; **le monter** fait l'inverse. Les courbes tracent ce compromis pour chaque appareil, avec le point correspondant à 0,5. Le seuil de 0,5 est un choix raisonnable, pas une obligation : pour piloter des équipements (flexibilité), on peut préférer un seuil plus élevé pour n'agir que lorsque le modèle est sûr de lui.

In [ ]:
ths = np.linspace(0.05, 0.95, 19)
m_te = te[i_test:]
fig, axes = plt.subplots(1, n_app, figsize=(3.4 * n_app, 3.0), sharey=True)
for ax, i, a in zip(np.atleast_1d(axes), range(n_app), APPLIANCES):
    t = (Yw[i, i_test:] > thr[i])[m_te]; pr = p_on[i][m_te]
    P, R, F = [], [], []
    for th in ths:
        p = pr > th; tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
        P.append(tp / (tp + fp) if tp + fp else np.nan); R.append(tp / (tp + fn) if tp + fn else np.nan)
        F.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan)
    ax.plot(ths, P, label="précision", color="#D99200"); ax.plot(ths, R, label="rappel", color="#0E7C86")
    ax.plot(ths, F, label="F1", color="#444", ls="--"); ax.axvline(0.5, color="#aaa", lw=1)
    ax.set_title(a, fontsize=10); ax.set_xlabel("seuil de probabilité"); ax.set_ylim(0, 1.02)
np.atleast_1d(axes)[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 11. Flexibilité : adapter la consommation au prix de l'électricité

**Demande du client :** montrer qu'on peut décaler une partie de la consommation vers les moments où l'électricité est moins chère, avec deux leviers : le **chauffe-eau** (stocker de la chaleur dans la cuve) et une **batterie locale**.

**Ce qui est mesuré, ce qui est simulé.** La consommation de la maison vient des **vraies mesures** UK-DALE de la période de test (107 jours complets, 9,3 kWh/jour en moyenne). La maison UK-DALE n'a pas de chauffe-eau électrique ni de batterie : je les **simule** avec des paramètres réalistes réglables en tête de cellule :
- chauffe-eau : cuve de 7 kWh de chaleur (≈ 200 L), résistance de 2 kW, besoin de 5 kWh/jour réparti sur les heures de puisage, 1 %/h de pertes, stock minimal garanti ;
- batterie : 5 kWh, 2,5 kW, rendement de 95 % à la charge et à la décharge, pas de revente au réseau.

**Les prix.** Deux signaux : un **tarif heures pleines / heures creuses** (0,27 / 0,21 €/kWh, heures creuses de 22 h à 6 h) et un **prix dynamique** horaire. Faute de vrais prix pour l'instant, le prix dynamique est *synthétique* mais de forme réaliste (pointes le matin et le soir, creux la nuit et en milieu de journée, 0,165 €/kWh en moyenne) ; il peut être remplacé par de vrais prix horaires avec un fichier `prix.csv` (colonnes `time,eur_kwh`).

**Les 4 scénarios**, comparés sur les mêmes jours et les mêmes prix :
1. **Référence** : chauffe-eau avec thermostat classique (il relance dès que la cuve descend sous 75 %, quel que soit le prix), pas de batterie ;
2. **Chauffe-eau piloté** : il chauffe aux heures les moins chères de la journée, en gardant toujours de l'eau chaude ;
3. **Batterie seule** : elle se charge quand l'électricité est bon marché et se décharge quand elle est chère ;
4. **Chauffe-eau + batterie**, optimisés ensemble.

Le pilotage est un petit problème d'optimisation linéaire (`scipy.optimize.linprog`) résolu **jour par jour, au pas horaire**, avec les prix du jour connus à l'avance (comme les prix « day-ahead » du marché). Le résultat est donc un **plafond de ce qu'on peut gagner**, pas ce qu'obtiendrait un vrai régulateur avec des prévisions imparfaites.

In [ ]:
from scipy.optimize import linprog

# ================== RÉGLAGES FLEXIBILITÉ (à adapter) ==================
PRIX_HP, PRIX_HC = 0.27, 0.21                 # €/kWh heures pleines / creuses (ordre de grandeur : à remplacer par ton tarif)
HC_HOURS = [22, 23, 0, 1, 2, 3, 4, 5]         # heures creuses (8 h par nuit)
PRIX_DYN_FILE = DATA_DIR / "prix.csv"         # optionnel : colonnes time,eur_kwh (une ligne par heure, mêmes dates que le test)
# Chauffe-eau (simulé)
WH_CAP = 7.0        # kWh de chaleur stockables dans la cuve (≈ 200 L, 30 °C d'écart)
WH_P = 2.0          # puissance de chauffe (kW)
WH_DAY = 5.0        # besoin d'eau chaude par jour (kWh)
WH_LOSS = 0.01      # pertes par heure (fraction du stock)
WH_MIN = 1.0        # stock minimal à tout moment (confort : il reste de l'eau chaude)
WH_LOW = 0.75       # thermostat classique : relance quand le stock passe sous 75 %
WH_END = 0.6        # le soir, on veut au moins 60 % de stock pour le lendemain matin
DRAW = np.zeros(24)
for h_, w_ in {6: .10, 7: .20, 8: .10, 12: .05, 13: .05, 18: .10, 19: .15, 20: .10, 21: .10, 22: .05}.items():
    DRAW[h_] = w_ * WH_DAY                    # soutirage d'eau chaude par heure (kWh)
# Batterie (simulée)
BAT_CAP, BAT_P, BAT_ETA = 5.0, 2.5, 0.95      # capacité kWh, puissance kW, rendement à la charge et à la décharge
BAT_SOC_MIN, BAT_START, BAT_END = 0.5, 2.5, 2.5   # stock minimal, de départ et de fin de journée (kWh)
SHIFT_APPS = ["washing_machine", "dishwasher"]    # appareils décalables (section 12)
SHIFT_WINDOW = (0, 24)                            # heures où l'on accepte de lancer un cycle décalé
# =====================================================================

# --- consommation horaire de la maison (vraies mesures, période de test) ---
s_agg = pd.Series(np.where(valid, agg, np.nan), index=data.index).iloc[i_test:]
hourly = s_agg.resample("1h").mean() / 1000.0                 # kWh consommés pendant chaque heure
by_day = hourly.groupby(hourly.index.date)
ok_days = [d for d, x in by_day if len(x) == 24 and x.isna().sum() <= 2]
if len(ok_days) < 3:
    raise SystemExit("Pas assez de jours complets dans le test pour simuler la flexibilité.")
hourly = hourly[[d in set(ok_days) for d in hourly.index.date]].ffill().bfill()
D = len(ok_days)
hours_idx = hourly.index
base = hourly.values.reshape(D, 24)                           # (jours, 24) kWh/h
print(f"{D} jours complets de test, du {ok_days[0]} au {ok_days[-1]} ; consommation moyenne {base.sum() / D:.1f} kWh/jour")

# --- signaux de prix (€/kWh) ---
hod = hours_idx.hour.values
price_hphc = np.where(np.isin(hod, HC_HOURS), PRIX_HC, PRIX_HP).reshape(D, 24)
if PRIX_DYN_FILE.exists():
    pr = pd.read_csv(PRIX_DYN_FILE, parse_dates=["time"]).set_index("time")["eur_kwh"]
    pser = pr.reindex(hours_idx).ffill().bfill()
    price_dyn = pser.values.reshape(D, 24); PRIX_DYN_SOURCE = "fichier prix.csv"
else:
    rng = np.random.default_rng(1)
    hh = np.arange(24)
    shape = (1 + 0.35 * np.exp(-((hh - 8) / 1.8) ** 2) + 0.60 * np.exp(-((hh - 19) / 2.0) ** 2)
             - 0.30 * np.exp(-((hh - 13.5) / 2.5) ** 2) - 0.25 * np.exp(-((hh - 3.5) / 3.0) ** 2))
    day_factor = np.exp(rng.normal(0, 0.25, size=(D, 1)))
    price_dyn = np.clip(0.16 * day_factor * shape[None, :] * (1 + rng.normal(0, 0.05, size=(D, 24))), 0.03, None)
    PRIX_DYN_SOURCE = "synthétique (à remplacer par de vrais prix)"
PRICES = {"HP/HC": price_hphc, "dynamique": price_dyn}
print("prix dynamique :", PRIX_DYN_SOURCE, f"| moyen {price_dyn.mean():.3f} €/kWh, min {price_dyn.min():.3f}, max {price_dyn.max():.3f}")

In [ ]:
def thermostat_day(e0, heating0):
    # chauffe-eau classique : relance sous WH_LOW, chauffe jusqu'au plein, sans regarder le prix
    h = np.zeros(24); e = e0; heating = heating0
    for t in range(24):
        e_pre = e * (1 - WH_LOSS) - DRAW[t]
        if e_pre < WH_LOW * WH_CAP: heating = True
        if heating:
            h[t] = min(WH_P, WH_CAP - e_pre)
            if e_pre + h[t] >= WH_CAP - 1e-9: heating = False
        e = e_pre + h[t]
    return h, e, heating

def optimise_day(price, base_d, e0, s0, wh_flex, bat_on, h_fixed):
    # programme linéaire sur 24 h : minimise le coût d'achat, sous contraintes de stock (cuve, batterie)
    n = 24; nv = 5 * n
    ih, ic, idd, ie, isoc = (slice(k * n, (k + 1) * n) for k in range(5))
    cost = np.concatenate([price, price, -price, np.zeros(n), np.zeros(n)])
    Aeq = np.zeros((2 * n, nv)); beq = np.zeros(2 * n)
    for t in range(n):
        Aeq[t, ie.start + t] = 1; Aeq[t, ih.start + t] = -1                 # e_t = (1-L) e_{t-1} + h_t - soutirage_t
        if t > 0: Aeq[t, ie.start + t - 1] = -(1 - WH_LOSS)
        beq[t] = DRAW[t] * -1 + ((1 - WH_LOSS) * e0 if t == 0 else 0)
        r = n + t
        Aeq[r, isoc.start + t] = 1; Aeq[r, ic.start + t] = -BAT_ETA; Aeq[r, idd.start + t] = 1 / BAT_ETA   # s_t = s_{t-1} + eta c - d/eta
        if t > 0: Aeq[r, isoc.start + t - 1] = -1
        beq[r] = s0 if t == 0 else 0
    Aub = np.zeros((n, nv)); bub = np.zeros(n)
    for t in range(n):
        Aub[t, idd.start + t] = 1; Aub[t, ih.start + t] = -1; bub[t] = base_d[t]   # on ne décharge que pour couvrir la consommation
    b = [(0, WH_P) if wh_flex else (h_fixed[t], h_fixed[t]) for t in range(n)]
    b += [(0, BAT_P) if bat_on else (0, 0) for _ in range(n)] * 2
    b += [(WH_MIN if wh_flex else 0, WH_CAP + 1e-6) for _ in range(n)]
    if wh_flex: b[3 * n + n - 1] = (max(WH_MIN, WH_END * WH_CAP), WH_CAP + 1e-6)
    b += [(BAT_SOC_MIN, BAT_CAP) if bat_on else (s0, s0) for _ in range(n)]
    if bat_on: b[4 * n + n - 1] = (max(BAT_SOC_MIN, BAT_END), BAT_CAP)
    res = linprog(cost, A_ub=Aub, b_ub=bub, A_eq=Aeq, b_eq=beq, bounds=b, method="highs")
    if res.status != 0: return None
    x = res.x
    return x[ih], x[ic], x[idd], x[ie], x[isoc]

def run_scenario(price_arr, wh_flex, bat_on):
    e, heating, s = WH_CAP, False, (BAT_START if bat_on else 0.0)
    H = np.zeros((D, 24)); C = np.zeros((D, 24)); Dis = np.zeros((D, 24)); S = np.zeros((D, 24)); fails = 0
    for k in range(D):
        h_th, e_th, heating_th = thermostat_day(e, heating)
        res = optimise_day(price_arr[k], base[k], e, s, wh_flex, bat_on, h_th) if (wh_flex or bat_on) else None
        if res is None:
            if wh_flex or bat_on: fails += 1
            H[k] = h_th; e, heating = e_th, heating_th
            S[k] = s
        else:
            H[k], C[k], Dis[k], e_arr, S[k] = res
            e = e_arr[-1]; s = S[k][-1] if bat_on else s
            heating = e < WH_LOW * WH_CAP
            if not wh_flex: e, heating = e_th, heating_th        # le thermostat garde son propre état
    load = base + H + C - Dis                                  # kWh achetés au réseau, par heure
    return {"H": H, "C": C, "D": Dis, "S": S, "load": load, "cost": float((load * price_arr).sum()), "fails": fails}

SCEN = [("1. Référence", False, False), ("2. Chauffe-eau piloté", True, False),
        ("3. Batterie seule", False, True), ("4. Chauffe-eau + batterie", True, True)]
sim = {}
rows = []
for pm, parr in PRICES.items():
    ref_cost = None
    for name, wf, bo in SCEN:
        r = run_scenario(parr, wf, bo); sim[(pm, name)] = r
        if ref_cost is None: ref_cost = r["cost"]
        rows.append({"prix": pm, "scénario": name, "coût sur la période (€)": r["cost"], "coût par jour (€)": r["cost"] / D,
                     "économie (€)": ref_cost - r["cost"], "économie (%)": 100 * (ref_cost - r["cost"]) / ref_cost,
                     "énergie achetée (kWh)": r["load"].sum(), "pic horaire (kW)": r["load"].max(),
                     "jours en secours": r["fails"]})
flex_res = pd.DataFrame(rows).set_index(["prix", "scénario"])
flex_res.round(2)

### Lire le tableau et les graphiques
- **Économie (%)** : gain par rapport au scénario 1 avec le **même signal de prix**. Avec le tarif HP/HC, seul le décalage vers la nuit rapporte ; avec un prix dynamique, il y a davantage de variations à exploiter.
- **Énergie achetée** : avec la batterie, j'achète un peu plus de kWh que la maison n'en consomme (pertes de charge et de décharge). L'économie vient du **prix** auquel l'énergie est achetée, pas du volume.
- **Pic horaire** : la puissance maximale tirée sur une heure. Tout décaler vers les heures creuses peut créer un **nouveau pic la nuit**, qu'il faudra surveiller (puissance souscrite).
- **Jours en secours** : jours où l'optimiseur n'aurait pas trouvé de solution et où je reviens au thermostat ; il doit valoir 0.
- La cellule donne aussi un ordre de grandeur annuel (×365/107). Ce n'est pas un gain « garanti » : les prix sont connus à l'avance, la batterie est parfaite (pas d'usure) et le coût du matériel n'est pas compté.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=False)
cols = ["#B0BEC5", "#0E7C86", "#D99200", "#5B3F8C"]
for ax, pm in zip(axes, PRICES):
    vals = [sim[(pm, n)]["cost"] / D for n, _, _ in SCEN]
    bars = ax.bar(range(4), vals, color=cols)
    for b_, v in zip(bars, vals): ax.text(b_.get_x() + b_.get_width() / 2, v, f"{v:.2f} €", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(4)); ax.set_xticklabels(["1\nréférence", "2\nchauffe-eau", "3\nbatterie", "4\nchauffe-eau\n+ batterie"], fontsize=8)
    ax.set_title(f"Coût d'électricité par jour — prix {pm}", fontsize=10); ax.set_ylabel("€ / jour")
    ax.set_ylim(0, max(vals) * 1.15)
plt.tight_layout(); plt.show()
print(f"Ordre de grandeur annuel (×365/{D}) :")
for pm in PRICES:
    ref = sim[(pm, SCEN[0][0])]["cost"] / D * 365
    for n, _, _ in SCEN[1:]:
        print(f"  prix {pm:10s} {n:28s} économie ≈ {ref - sim[(pm, n)]['cost'] / D * 365:6.0f} € / an")

### Une journée type : qui fait quoi
J'ai choisi le jour où l'écart de prix dynamique entre l'heure la plus chère et la moins chère est le plus grand. En haut, le **prix**. Au milieu, la **puissance du chauffe-eau** : le thermostat (scénario 1) chauffe quand la cuve se vide, alors que le pilotage (scénarios 2 et 4) déplace la chauffe vers les heures bon marché. En bas, la **batterie** (scénario 4) : barres vers le haut pour la charge, vers le bas pour la décharge, et la courbe donne son niveau de charge.

In [ ]:
pm = "dynamique"
k = int(np.argmax(price_dyn.max(1) - price_dyn.min(1)))
hh = np.arange(24)
r1, r2, r4 = sim[(pm, SCEN[0][0])], sim[(pm, SCEN[1][0])], sim[(pm, SCEN[3][0])]
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].step(hh, price_dyn[k], where="post", color="#444"); axes[0].set_ylabel("€/kWh"); axes[0].set_title(f"Prix dynamique, {ok_days[k]}", fontsize=10)
axes[1].step(hh, r1["H"][k], where="post", color="#B0BEC5", lw=2, label="1. thermostat")
axes[1].step(hh, r2["H"][k], where="post", color="#0E7C86", label="2. piloté")
axes[1].step(hh, r4["H"][k], where="post", color="#5B3F8C", ls="--", label="4. piloté + batterie")
axes[1].set_ylabel("kW chauffe-eau"); axes[1].legend(fontsize=8, loc="upper right")
axes[2].bar(hh, r4["C"][k] - r4["D"][k], color=np.where(r4["C"][k] - r4["D"][k] >= 0, "#D99200", "#C0392B"), width=0.8, align="edge")
axes[2].set_ylabel("kW batterie (+ charge, − décharge)")
ax2 = axes[2].twinx(); ax2.plot(hh + 1, r4["S"][k], color="#333"); ax2.set_ylabel("niveau (kWh)"); ax2.set_ylim(0, BAT_CAP); ax2.grid(False)
axes[2].set_xlabel("heure de la journée"); axes[2].set_xticks(range(0, 25, 3))
plt.tight_layout(); plt.show()

## 12. Ce que la désagrégation apporte à la flexibilité
Le chauffe-eau et la batterie se pilotent **sans** savoir ce que font les autres appareils. En revanche, le lave-linge et le lave-vaisselle sont eux aussi décalables, et pour les décaler il faut **savoir quand ils tournent et combien ils consomment** : c'est précisément le rôle du modèle NILM.

Pour ces deux appareils, je compare donc l'énergie qu'ils consomment **vraiment**, celle que **le réseau** ou **le CO** leur attribue, et l'économie **potentielle** si, chaque jour, toute cette énergie était déplacée à l'heure la moins chère de la plage autorisée (`SHIFT_WINDOW`). C'est un plafond simplifié (un cycle dure plusieurs heures et l'utilisateur doit être d'accord), mais il répond à la question : **la désagrégation est-elle assez fiable pour chiffrer la flexibilité ?** Si l'économie estimée à partir du modèle est proche de l'économie réelle, oui ; sinon, il faudrait la compléter par une mesure directe (prise connectée).

In [ ]:
def hourly_app(sig, a):
    i = APPLIANCES.index(a)
    s = pd.Series(np.where(valid, sig[i], np.nan), index=data.index).iloc[i_test:]
    return (s.resample("1h").mean() / 1000.0).reindex(hours_idx).fillna(0).values.reshape(D, 24)

w0, w1 = SHIFT_WINDOW
rows = []
for pm, parr in PRICES.items():
    pmin = parr[:, w0:w1].min(1)                                  # prix le plus bas de la fenêtre, chaque jour
    for a in [x for x in SHIFT_APPS if x in APPLIANCES]:
        for src, sig in [("réel", Yw), ("réseau", pred_nn), ("CO", pred_co)]:
            e = hourly_app(sig, a)
            now = float((e * parr).sum()); shifted = float((e.sum(1) * pmin).sum())
            rows.append({"prix": pm, "appareil": a, "source": src, "énergie (kWh)": e.sum(),
                         "coût actuel (€)": now, "économie potentielle (€)": now - shifted})
shift_res = pd.DataFrame(rows).set_index(["prix", "appareil", "source"])
shift_tab = shift_res.round(2)
print("Économie potentielle sur la période de test, si tout est décalé à l'heure la moins chère :")
shift_tab

In [ ]:
tab = shift_res["économie potentielle (€)"].unstack("source")[["réel", "réseau", "CO"]]
tab["écart réseau (%)"] = 100 * (tab["réseau"] - tab["réel"]) / tab["réel"].where(tab["réel"].abs() > 1e-9)
tab["écart CO (%)"] = 100 * (tab["CO"] - tab["réel"]) / tab["réel"].where(tab["réel"].abs() > 1e-9)
print("Économie potentielle en € (et écart par rapport au réel) :")
tab.round(2)

## 13. Résultats et analyse
**Désagrégation (section 8)** : le réseau de base obtient sur le test un F1 de 0,90 pour le lave-vaisselle, 0,88 pour le lave-linge, 0,80 pour le frigo et 0,69 pour la bouilloire, contre 0,06 à 0,50 pour le CO, avec une erreur d'énergie de 15 à 28 %.

**Matrices de confusion (section 10)** : la diagonale du réseau est nettement plus foncée que celle du CO. Le CO produit énormément de **fausses alertes** sur le lave-vaisselle et le lave-linge (il les « allume » à chaque pic de la bouilloire ou d'un appareil non suivi), alors que le réseau en produit très peu. Pour le frigo, le CO **rate** une grande partie des cycles, que le réseau retrouve.

**Seuil** : quand la précision est faible mais le rappel bon (cas de la bouilloire), monter le seuil au-dessus de 0,5 réduit les fausses alertes ; pour piloter des équipements, je préfère un seuil prudent.

**Flexibilité (section 11)** : sur les 107 jours de test, le pilotage du chauffe-eau et de la batterie fait économiser environ **16 %** de la facture avec le tarif HP/HC et environ **33 %** avec le prix dynamique synthétique (scénario 4, par rapport au thermostat classique). Le chauffe-eau piloté apporte l'essentiel du gain en HP/HC ; la batterie devient intéressante quand les prix varient beaucoup dans la journée. Ces chiffres sont un ordre de grandeur (prix synthétiques et connus à l'avance) : ils seront à refaire avec de vrais prix (`prix.csv`).

**Désagrégation et flexibilité (section 12)** : si l'économie estimée à partir du réseau est proche de l'économie réelle pour le lave-linge et le lave-vaisselle, le NILM suffit pour chiffrer la flexibilité de ces appareils ; celle estimée avec le CO est très surévaluée, car il leur attribue beaucoup trop d'énergie.

In [ ]:
# sauvegardes pour le rapport (dans le panneau de fichiers de Colab)
results.round(3).to_csv("resultats_nilm.csv"); flex_res.round(3).to_csv("resultats_flexibilite.csv"); shift_res.round(3).to_csv("resultats_decalables.csv")
print("fichiers écrits : resultats_nilm.csv, resultats_flexibilite.csv, resultats_decalables.csv")

## 14. Conclusion
- **Modèle** : réseau convolutif 1D dilaté (32 canaux, 7 blocs résiduels de dilatations 1 à 64 min, 22 120 paramètres, ≈ 22 Ko en int8), qui voit ±127 minutes autour de chaque minute.
- **Entrées** : puissance totale de la maison en kW + heure de la journée (sinus, cosinus), une valeur par minute.
- **Sorties** : pour chaque minute et chacun des 4 appareils, une puissance (W) et un état allumé / éteint (probabilité > 0,5).
- **Performances** : F1 de 0,69 à 0,90 contre 0,06 à 0,50 pour le CO ; les matrices de confusion montrent que le gain vient surtout de la quasi-disparition des fausses alertes sur les appareils à cycles.
- **Intérêt** : le modèle fournit l'information qui manque pour la flexibilité, c'est-à-dire **quand** tournent les appareils décalables et **combien** ils consomment. La simulation montre l'enjeu : 16 % (HP/HC) à 33 % (prix dynamique) d'économie avec un chauffe-eau et une batterie pilotés.
- **Limites** : une seule maison réelle pour apprendre ; flexibilité calculée avec des prix synthétiques connus à l'avance et des équipements simulés.
- **Suite** : semaine 5, ajouter des foyers simulés (LoadProfileGenerator) pour donner plus de variété à l'entraînement.